# 3. Train the anomaly model

The public PPIP snapshot does not provide verified integrity outcome labels. This project therefore does not train a supervised fraud classifier. It uses Isolation Forest to rank unusual tender profiles for further review.

The first 70% of dated tenders form the training period. The later 30% are held back for a chronological check. Preprocessing is fitted on the training period only.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import mlflow
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from IPython.display import display

PROJECT = Path.cwd().resolve()
if not (PROJECT / "data").exists():
    PROJECT = PROJECT.parent
FEATURE_FILE = PROJECT / "data" / "processed" / "ppip_features.csv"
MODEL_DIR = PROJECT / "artifacts"
REPORT_DIR = PROJECT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
features = pd.read_csv(FEATURE_FILE, parse_dates=["tender_start_date"])
INPUT_COLUMNS = [col for col in features.columns if col not in ("ocid", "tender_start_date")]

## Split the data by date

In [ ]:
features = features.sort_values(["tender_start_date", "ocid"]).reset_index(drop=True)
split_date = features["tender_start_date"].quantile(0.70)
train = features[features["tender_start_date"] <= split_date].copy()
holdout = features[features["tender_start_date"] > split_date].copy()
if train.empty or holdout.empty or train["tender_start_date"].max() >= holdout["tender_start_date"].min():
    raise AssertionError("Could not create a strictly chronological train/holdout split.")

NUMERIC_COLUMNS = [col for col in INPUT_COLUMNS if pd.api.types.is_numeric_dtype(features[col])]
TEXT_COLUMN = "tender_text" if "tender_text" in INPUT_COLUMNS else None
CATEGORICAL_COLUMNS = [col for col in INPUT_COLUMNS if col not in NUMERIC_COLUMNS and col != TEXT_COLUMN]
print("Training rows:", f"{len(train):,}")
print("Later holdout rows:", f"{len(holdout):,}")
print("Training ends:", train["tender_start_date"].max())
print("Holdout starts:", holdout["tender_start_date"].min())

## Build the preprocessing and model pipeline

Numeric values are median-imputed and scaled. Categories are imputed and one-hot encoded. Tender titles and descriptions are represented with TF-IDF. The whole preprocessing step is part of the pipeline, so the same transformations are used when the API scores new records.

In [ ]:
def make_preprocessor():
    transformers = []
    if NUMERIC_COLUMNS:
        numeric = Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", RobustScaler()),
        ])
        transformers.append(("numeric", numeric, NUMERIC_COLUMNS))
    if CATEGORICAL_COLUMNS:
        categorical = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("categorical", categorical, CATEGORICAL_COLUMNS))
    if TEXT_COLUMN:
        text = TfidfVectorizer(
            max_features=2500,
            min_df=2,
            max_df=0.98,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
        )
        transformers.append(("text", text, TEXT_COLUMN))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.3)


pipeline = Pipeline([
    ("preparation", make_preprocessor()),
    ("model", IsolationForest(
        n_estimators=300,
        contamination=0.05,
        max_samples="auto",
        random_state=42,
        n_jobs=-1,
    )),
])

pipeline.fit(train[INPUT_COLUMNS])
train_scores = -pipeline.decision_function(train[INPUT_COLUMNS])
holdout_scores = -pipeline.decision_function(holdout[INPUT_COLUMNS])
reference_scores = np.sort(train_scores)
holdout_percentiles = np.searchsorted(reference_scores, holdout_scores, side="right") / len(reference_scores)

## Check whether the ranking changes with the random seed

There are no verified labels to calculate accuracy, precision, recall or F1. Instead, the notebook checks whether the ordering is broadly similar when Isolation Forest is fitted with different random seeds. This is a stability check, not an accuracy measure.

In [ ]:
X_train = pipeline.named_steps["preparation"].transform(train[INPUT_COLUMNS])
X_holdout = pipeline.named_steps["preparation"].transform(holdout[INPUT_COLUMNS])
seed_scores = [holdout_scores]
for seed in (43, 44):
    candidate = IsolationForest(
        n_estimators=300, contamination=0.05, max_samples="auto",
        random_state=seed, n_jobs=-1,
    )
    candidate.fit(X_train)
    seed_scores.append(-candidate.decision_function(X_holdout))

rank_correlations = []
for left in range(len(seed_scores)):
    for right in range(left + 1, len(seed_scores)):
        corr = spearmanr(seed_scores[left], seed_scores[right]).statistic
        if np.isfinite(corr):
            rank_correlations.append(float(corr))
mean_rank_stability = float(np.mean(rank_correlations)) if rank_correlations else np.nan
print("Mean pairwise Spearman rank correlation:", round(mean_rank_stability, 4))

## Save the model and record the experiment

In [ ]:
scored = holdout[["ocid", "tender_start_date"]].copy()
scored["anomaly_score"] = holdout_scores
scored["anomaly_percentile"] = holdout_percentiles
scored["review_band"] = np.select(
    [scored["anomaly_percentile"] >= 0.98, scored["anomaly_percentile"] >= 0.90],
    ["high", "review"], default="routine",
)
scored = scored.sort_values("anomaly_percentile", ascending=False).reset_index(drop=True)
scored.to_csv(REPORT_DIR / "holdout_scores.csv", index=False)

SOURCE_HASH_FILE = PROJECT / "data" / "processed" / "source_sha256.txt"
source_hash = SOURCE_HASH_FILE.read_text().strip() if SOURCE_HASH_FILE.exists() else "not-recorded"
model_bundle = {
    "pipeline": pipeline,
    "reference_scores": reference_scores,
    "input_columns": INPUT_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "text_column": TEXT_COLUMN,
    "source_sha256": source_hash,
    "train_start": str(train["tender_start_date"].min()),
    "train_end": str(train["tender_start_date"].max()),
    "model_version": "1.0",
}
MODEL_PATH = MODEL_DIR / "ppip_anomaly_model.joblib"
joblib.dump(model_bundle, MODEL_PATH)

summary = pd.DataFrame([
    ("model", "Isolation Forest with numeric, categorical and TF-IDF features"),
    ("training_rows", len(train)),
    ("holdout_rows", len(holdout)),
    ("training_start", str(train["tender_start_date"].min())),
    ("training_end", str(train["tender_start_date"].max())),
    ("holdout_start", str(holdout["tender_start_date"].min())),
    ("holdout_end", str(holdout["tender_start_date"].max())),
    ("mean_rank_stability_spearman", mean_rank_stability),
    ("holdout_share_above_90th_training_percentile", float((holdout_percentiles >= 0.90).mean())),
    ("holdout_share_above_98th_training_percentile", float((holdout_percentiles >= 0.98).mean())),
    ("verified_outcome_labels", "not available"),
], columns=["item", "value"])
summary.to_csv(REPORT_DIR / "model_summary.csv", index=False)

mlflow.set_tracking_uri(f"sqlite:///{PROJECT / 'mlflow.db'}")
mlflow.set_experiment("PPIP Procurement Anomaly Screening")
with mlflow.start_run(run_name="isolation_forest_temporal_holdout"):
    mlflow.log_param("model", "IsolationForest")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("contamination", 0.05)
    mlflow.log_param("training_fraction", 0.70)
    mlflow.log_param("numeric_features", ",".join(NUMERIC_COLUMNS))
    mlflow.log_param("categorical_features", ",".join(CATEGORICAL_COLUMNS))
    mlflow.log_metric("training_rows", len(train))
    mlflow.log_metric("holdout_rows", len(holdout))
    if np.isfinite(mean_rank_stability):
        mlflow.log_metric("mean_rank_stability_spearman", mean_rank_stability)
    mlflow.log_metric("share_above_90th_training_percentile", float((holdout_percentiles >= 0.90).mean()))
    mlflow.log_artifact(str(MODEL_PATH))

print("Saved model:", MODEL_PATH)
display(summary)
display(scored.head(10))